# ĐỒ ÁN PHÂN TÍCH DỮ LIỆU
## QUY TRÌNH 2.2 & 2.3: TIỀN XỬ LÝ DỮ LIỆU - TÍCH HỢP & BIẾN ĐỔI, THU GIẢM DỮ LIỆU
- **Thành viên thực hiện:** Điểu Xuân Hoàng
- **Nhiệm vụ theo phân công:**
  1. Tích hợp dữ liệu (Data Integration - Left join bảng Fact với Calendar, Subgroup, Group).
  2. Biến đổi và thu giảm dữ liệu (Data Transformation & Reduction - Chọn thuộc tính quan trọng, tạo bảng P&L theo Tháng x Dòng sản phẩm).

In [ ]:
# Kết nối Google Drive nếu đang chạy trên Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # Chạy trên môi trường cục bộ (Jupyter Notebook / VS Code)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Đường dẫn đến thư mục dữ liệu đã làm sạch
PROCESSED = Path("DATA_PROCESS/data processed") if Path("DATA_PROCESS/data processed").exists() else Path("/content/drive/MyDrive/[LTPTDL] Đồ án/data processed")

FILES = [
    "Fact", "Calendar", "Business Line", "Group", "Subgroup",
    "Channel", "Region", "Headlines", "KPI", "Ratios"
]

# Đọc các bảng dữ liệu đã được làm sạch
tables = {
    f: pd.read_csv(PROCESSED / f"{f}.csv", dtype=str, encoding="utf-8-sig")
    for f in FILES
}

df = tables["Fact"].copy()
calendar = tables["Calendar"].copy()

# Ép kiểu dữ liệu phù hợp
df["Amount"] = df["Amount, $"].astype(float)
df["Date"] = pd.to_datetime(df["Date"])
calendar["Date"] = pd.to_datetime(calendar["Date"])

df = df.rename(columns={
    "Business Line ID": "BL_ID",
    "Business Line Name": "BL_Name",
    "Subgroup ID": "Subgroup_ID",
    "Group ID": "Group_ID",
    "Revenue or expense": "RevExp"
})

print("Kích thước bảng Fact đã làm sạch:", df.shape)
print("Kích thước bảng Calendar đã làm sạch:", calendar.shape)

## 2.2. Data integration

Mục đích: Kết hợp bảng Fact đã làm sạch với các bảng danh mục và Calendar nhằm bổ sung thông tin về khoản mục tài chính, nhóm tài chính và thời gian phục vụ phân tích. Nhóm sử dụng left join để bảo toàn số lượng giao dịch, đồng thời kiểm tra tính toàn vẹn của dữ liệu sau khi tích hợp.

In [ ]:
# 1. Chuẩn bị bảng danh mục
sg = tables["Subgroup"][["Subgroup ID", "Subgroup"]].astype(str)
gr = tables["Group"][["Group ID", "Group"]].astype(str)
sg.columns = ["Subgroup_ID", "Subgroup_Name"]
gr.columns = ["Group_ID", "Group_Name"]

# 2. Tích hợp dữ liệu (left join, không làm mất giao dịch)
integrated = (
    df.merge(calendar[["Date", "Year", "Quarter", "Month", "Month in text"]],
             on="Date", how="left", validate="many_to_one")
      .merge(sg, on="Subgroup_ID", how="left", validate="many_to_one")
      .merge(gr, on="Group_ID", how="left", validate="many_to_one")
)

# 3. Đổi tên cột dòng sản phẩm
integrated = integrated.rename(columns={"BL_Name": "Business_Line"})

In [ ]:
#Kiểm tra kết quả Data Integration
pd.Series({
    "Số dòng trước tích hợp": len(df),
    "Số dòng sau tích hợp": len(integrated),
    "Calendar đầy đủ": len(calendar),
    "Subgroup chưa ghép được": integrated["Subgroup_Name"].isna().sum(),
    "Group chưa ghép được": integrated["Group_Name"].isna().sum(),
    "Ngày chưa ghép được": integrated["Month"].isna().sum()
})

,0
Số dòng trước tích hợp,8515
Số dòng sau tích hợp,8515
Calendar đầy đủ,365
Subgroup chưa ghép được,0
Group chưa ghép được,0
Ngày chưa ghép được,0


Nhận xét: Quá trình tích hợp đã bảo toàn toàn bộ 8.515 giao dịch, không phát sinh tình trạng nhân bản hoặc mất dòng sau khi JOIN. Các thuộc tính thời gian, Subgroup và Group đều được liên kết thành công, không có giá trị thiếu. Kết quả cho thấy dữ liệu đã đảm bảo tính toàn vẹn khi tích hợp và sẵn sàng cho bước tiếp theo.

## 2.3. Data transformation & reduction

In [ ]:
# Chọn các thuộc tính cần thiết cho RQ1–RQ3
data = integrated[[
    "TransactionID", "Date", "Year", "Month", "Quarter",
    "Business_Line", "Subgroup_ID", "Group_ID",
    "Amount", "Channel", "Region"
]].copy()

# Chuyển chi phí sang giá trị dương để thuận tiện tính tỷ số
data["Amount_abs"] = data["Amount"].abs()

# Phân loại các khoản tài chính
data["Financial_Type"] = data["Group_ID"].map({
    "1": "Revenue",
    "2": "Opex",
    "3": "COGS",
    "4": "Interest_Tax"
})

In [ ]:
# Bảng P&L theo tháng x dòng sản phẩm (dùng chung cho RQ1, RQ2, RQ3)
pl = data.pivot_table(
    index=["Month", "Quarter", "Business_Line"],
    columns="Financial_Type", values="Amount_abs",
    aggfunc="sum", fill_value=0
).reset_index()
pl.columns.name = None

pl["Gross_Profit"] = pl["Revenue"] - pl["COGS"]
pl["GP_Margin"] = pl["Gross_Profit"] / pl["Revenue"]    # biên lợi nhuận gộp (RQ1)
pl["Opex_Ratio"] = pl["Opex"] / pl["Revenue"]           # tỷ lệ Opex / doanh thu (RQ2)

# Kiểm tra: không dòng nào bị bỏ sót khi gộp (kỳ vọng 36 dòng, hiệu = 0)
print("Số dòng pl:", len(pl),
      "| Chênh lệch tổng tiền:", round(data["Amount_abs"].sum() - pl[["Revenue", "COGS", "Opex", "Interest_Tax"]].sum().sum(), 2))

Số dòng pl: 36 | Chênh lệch tổng tiền: 0.0


 Nhận xét: Dữ liệu được tổng hợp thành bảng P&L gồm 36 quan sát theo tháng và dòng sản phẩm (Month × Business Line). Chênh lệch tổng giá trị tuyệt đối của Amount trước và sau tổng hợp bằng 0 cho thấy quá trình này đã bảo toàn tổng giá trị tài chính, đồng thời tạo ra các chỉ tiêu cần thiết phục vụ cho việc phân tích tiếp theo.

In [ ]:
# Xuất bảng P&L phục vụ phân tích các câu hỏi nghiên cứu RQ1, RQ2, RQ3
pl_path = PROCESSED / "pl.csv"
pl.to_csv(pl_path, index=False, encoding="utf-8-sig")
print("-> Đã lưu bảng P&L thành công tại:", pl_path)